# Tutorial
This file introduces solving the PDEs using the Dedalus framework.
Let's start by adding the library of Dedalus

In [ ]:
import dedalus.public as d3
# import numpy as np # add Numpy package
# import matplotlib.pyplot as plt # add package for plotting

## Coordinates
As traditional CFD methods, we need to create a domain space to solve PDEs on that. In Dedalus, we can create a grid system to do it. Cartesian grid system will be primarily indicated in this notebook. So, to create a 3D coordinates, we have the 'CartesianCoordinates' feature and define the name of coordinates. Specifically, for 1D problem, we can use another feature 'Coordinate' instead.

In [ ]:
coords = d3.CartesianCoordinates('x','y','z') # for 3D problem
# coords = d3.CartesianCoordinates('x','y') # for 2D problem
# xcoord = d3.Coordinate('x') # for 1D problem

## Distributor
Distributor is a feature to object direct the parallel decomposition of fields in all problems, including both single and multiple cores. In this feature, we also define the type of data fields via 'dtype' feature

In [ ]:
dtype = np.float64 # define type of data: np.float64 = double; np.float32 = float
dist = d3.Distributor(coords, dtype=dtype) # create distributor of problem based 
# on the Cartesian coordinates 'coords' and define the type 'float64' for data fields
# No mesh for serial / automatic parallelization

By default, problems are distributed across a 1-dimensional mesh of all available MPI processes. If you want to indicate a specific domain containing subdomains, let's use the feature 'mesh' to define domain. Normally, this is not recommended to perform by users. Pls read [https://dedalus-project.readthedocs.io/en/latest/notebooks/dedalus_tutorial_1.html] to learn more details.

## Bases
Before accessing problems, we have to create a basis. In Dedalus, there are many basis classes specifically designed for different problems. For example:
- RealFourier: for real periodic functions on an interval using cosine & sine modes.
- ComplexFourier: for complex periodic functions on an interval using complex exponentials.
- Chebyshev: for functions on an interval.
- ...

The multidimensional bases are instantiated with:
- The corresponding coordinate system,
- The multidimensional mode shape for the basis,
- The radial extent of the basis,
- The problem dtype

To specify dealiasing scale factors for each basis axis or all axis, we can use 'dealias'. To properly dealias quadratic nonlinearities, you would need a scaling factor of 3/2. To see difference between scaling factors of 1 and 3/2, pls read 'Basis grids and scale factors' section in [https://dedalus-project.readthedocs.io/en/latest/notebooks/dedalus_tutorial_1.html]

In [ ]:
xbasis = d3.RealFourier(coords['x'], size=32, bounds=(0,1), dealias=3/2)
ybasis = d3.RealFourier(coords['y'], size=32, bounds=(0,1), dealias=3/2)
zbasis = d3.Chebyshev(coords['z'], size=32, bounds=(0,1), dealias=3/2)
# size = N : N nodes on coordinate
# bounds=(A,B): bound of axis from A to B; e.g. bounds=(-5.2,10.5)
# dealias= X: the scaling factor with default of 1; = 3/2 for quadractic nonlinearity

To summary this notebook, we let see following code instantiating basis of 2D Rayleigh-Benard convection's problem:

In [ ]:
# add libraries
import numpy as np
import dedalus.public as d3

# define parameters of problem
Xmin, Xmax = 0, 4 # define domain with two axises x and z
Zmin, Zmax = 0, 1 
Nx, Nz = 256, 64 # define the number of nodes
Double = np.float64 
dealias = 3/2

# define basis of problem
coords = d3.CartesianCoordinates('x', 'z')
dist = d3.Distributor(coords, dtype=Double)
xbasis = d3.RealFourier(coords['x'], size=Nx, bounds=(Xmin, Xmax), dealias=dealias)
zbasis = d3.ChebyshevT(coords['z'], size=Nz, bounds=(Zmin, Zmax), dealias=dealias)